# C-MAPSS FD001–FD004 — Tune one combined XGBoost model

**Goal:** Optimize one XGBoost model across all four datasets, compare it fairly with our previous combined and four-specialist approaches, and log results **both locally and to DagsHub MLflow**.

**No need to rerun the previous notebook.** This notebook reads the existing curated Parquet files and starts a fresh tuning experiment.

**Evaluation rules**
- Split by engine within each FD dataset, using the same `random_state=42` and 20% validation split as the previous notebook.
- Tune hyperparameters with **3-fold engine-group cross-validation on the training portion only**.
- Compare the tuned model against the earlier models on the **same untouched validation portion**.
- **Do not use NASA test sets for tuning or model selection.** We will do the final test after choosing the model.
- Local logs and DagsHub MLflow runs are separate; the notebook verifies the remote tracking URI explicitly.

## 1. Imports, paths, and local logging

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from itertools import product
import csv
import json
import logging
import os
import joblib
import numpy as np
import pandas as pd
import mlflow
import dagshub
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, ParameterSampler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

# Locate the project root from either notebooks/ or the repository root.
ROOT = Path("..").resolve() if Path("../data/curated/train").exists() else Path(".").resolve()
DATA_DIR = ROOT / "data" / "curated" / "train"
DATASETS = ["FD001", "FD002", "FD003", "FD004"]
SEED = 42
N_TRIALS = 12
CV_FOLDS = 3

# Every execution gets its own folder; previous logs and models remain untouched.
RUN_NAME = datetime.now(timezone.utc).strftime("combined_xgb_%Y%m%d_%H%M%S")
RUN_DIR = ROOT / "reports" / "combined_xgboost_tuning" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
TRIALS_CSV = RUN_DIR / "cv_trials.csv"

# Log progress to a local file AND to the notebook output.
logger = logging.getLogger("combined_xgb_tuning")
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(RUN_DIR / "tuning.log", encoding="utf-8")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)
stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)
logger.propagate = False

logger.info("Local experiment folder: %s", RUN_DIR)
logger.info("Curated training folder exists: %s", DATA_DIR.exists())
assert DATA_DIR.is_dir(), "Cannot find data/curated/train. Pull your DVC data first."

## 2. Load all four datasets and recreate the original engine-level validation split

In [ ]:
# Load only TRAIN data; NASA test data is intentionally not loaded here.
raw_train = {fd: pd.read_parquet(DATA_DIR / f"train_{fd}.parquet") for fd in DATASETS}

# Match the exact feature schema used by our previous individual-vs-combined notebook.
base_features = [c for c in raw_train["FD001"].columns if c not in ["unit_number", "RUL"]]
indicator_features = [f"dataset_{fd}" for fd in DATASETS]
feature_columns = base_features + indicator_features

train_parts = []
val_parts = []
for fd in DATASETS:
    df = raw_train[fd].copy()
    assert {"unit_number", "time_in_cycles", "RUL"}.issubset(df.columns), f"Missing required columns in {fd}"
    assert set(base_features).issubset(df.columns), f"Feature mismatch in {fd}"
    for name in DATASETS:
        df[f"dataset_{name}"] = int(fd == name)
    df["dataset_id"] = fd
    df["engine_key"] = fd + ":" + df["unit_number"].astype(str)

    # Same split as the previous notebook: 80% of engines train, 20% validate.
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    train_idx, val_idx = next(splitter.split(df, groups=df["unit_number"]))
    train_df = df.iloc[train_idx].copy()
    val_df = df.iloc[val_idx].copy()
    assert set(train_df["engine_key"]).isdisjoint(set(val_df["engine_key"]))
    train_parts.append(train_df)
    val_parts.append(val_df)
    logger.info("%s: %s training engines, %s validation engines", fd, train_df["unit_number"].nunique(), val_df["unit_number"].nunique())

train_df = pd.concat(train_parts, ignore_index=True)
val_df = pd.concat(val_parts, ignore_index=True)
X_train = train_df[feature_columns]
y_train = train_df["RUL"]
X_val = val_df[feature_columns]
y_val = val_df["RUL"]

assert X_train.columns.tolist() == X_val.columns.tolist()
assert X_train.notna().all().all() and X_val.notna().all().all(), "Missing feature values detected."
logger.info("Training rows: %s | Validation rows: %s | Features: %s", len(X_train), len(X_val), len(feature_columns))

## 3. Connect to **DagsHub**, not a local SQLite MLflow database

This is the crucial fix from the previous notebook. We set and **verify** the remote tracking URL *before* creating any MLflow runs.

If authentication is needed, DagsHub may ask you to sign in. Local CSV/log files will still be saved if remote logging fails.

In [ ]:
# Initialize DagsHub authentication and point MLflow at the REMOTE server.
dagshub.init(repo_owner="Omesham", repo_name="predictive-maintenance-mlops", mlflow=True)
REMOTE_URI = "https://dagshub.com/Omesham/predictive-maintenance-mlops.mlflow/"
mlflow.set_tracking_uri(REMOTE_URI)

# Never accidentally write this experiment into notebooks/mlflow.db.
assert mlflow.get_tracking_uri().rstrip("/") == REMOTE_URI.rstrip("/"), "MLflow is NOT pointing at DagsHub."
mlflow.set_experiment("FD001_FD004_Combined_XGBoost_Tuning")
logger.info("MLflow tracking URI: %s", mlflow.get_tracking_uri())

## 4. Hyperparameter search using 3-fold grouped cross-validation

**What we're tuning:** number of trees, tree depth, learning rate, subsampling, and regularization.

**Numeric example:** If an engine has 180 cycles, all 180 rows for that engine stay together in one fold. The model never trains on cycle 1 of an engine and validates on cycle 180 of that *same* engine.

**Runtime:** 12 configurations × 3 folds = **36 XGBoost fits**, plus one validation fit and the final deployment refit. This can take several minutes. Each finished trial is written to a local CSV immediately and logged to MLflow. To start smaller, change `N_TRIALS` to 6 before running.

In [ ]:
# Keep parameter choices explicit and reproducible.
search_space = {
    "n_estimators": [200, 350, 500],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.03, 0.05, 0.1],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "reg_lambda": [1.0, 5.0, 10.0],
}
candidates = list(ParameterSampler(search_space, n_iter=N_TRIALS, random_state=SEED))

# Engine IDs overlap across FD datasets; dataset-prefixed keys make them unique.
cv = GroupKFold(n_splits=CV_FOLDS)
cv_indices = list(cv.split(X_train, y_train, groups=train_df["engine_key"]))

# Confirm that no engine appears on both sides of a CV fold.
for fit_idx, score_idx in cv_indices:
    assert set(train_df.iloc[fit_idx]["engine_key"]).isdisjoint(set(train_df.iloc[score_idx]["engine_key"]))

def build_xgb(params):
    # A single new model per fold; n_jobs uses local CPU cores.
    return XGBRegressor(**params, objective="reg:squarederror", random_state=SEED, n_jobs=-1, tree_method="hist")

def score_predictions(actual, predicted):
    return {
        "mae": float(mean_absolute_error(actual, predicted)),
        "rmse": float(np.sqrt(mean_squared_error(actual, predicted))),
    }

logger.info("Prepared %s configurations and %s engine-safe folds.", len(candidates), len(cv_indices))

## 5. Run tuning — save every result locally and log each trial remotely

In [ ]:
# This is a NEW tuning experiment. Do not rerun this cell unless you intend a new search.
trial_records = []
for trial_no, params in enumerate(candidates, start=1):
    fold_mae = []
    fold_rmse = []
    logger.info("Starting trial %s/%s: %s", trial_no, len(candidates), params)

    for fold_no, (fit_idx, score_idx) in enumerate(cv_indices, start=1):
        estimator = build_xgb(params)
        estimator.fit(X_train.iloc[fit_idx], y_train.iloc[fit_idx])
        predictions = estimator.predict(X_train.iloc[score_idx])
        scores = score_predictions(y_train.iloc[score_idx], predictions)
        fold_mae.append(scores["mae"])
        fold_rmse.append(scores["rmse"])
        logger.info("Trial %s fold %s: MAE %.3f, RMSE %.3f", trial_no, fold_no, scores["mae"], scores["rmse"])
        del estimator

    record = {
        "trial": trial_no,
        **params,
        "cv_mae": float(np.mean(fold_mae)),
        "cv_rmse": float(np.mean(fold_rmse)),
        "cv_mae_std": float(np.std(fold_mae)),
    }
    trial_records.append(record)

    # Write the completed trial to a local CSV immediately, even if DagsHub is unavailable.
    with TRIALS_CSV.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(record.keys()))
        if handle.tell() == 0:
            writer.writeheader()
        writer.writerow(record)

    # Log metrics and parameters, NOT large model artifacts, for each trial.
    try:
        with mlflow.start_run(run_name=f"{RUN_NAME}_trial_{trial_no:02d}"):
            mlflow.set_tags({"stage": "cross_validation", "approach": "combined", "dataset": "FD001-FD004", "local_run_name": RUN_NAME})
            mlflow.log_params({**params, "random_state": SEED, "cv_folds": CV_FOLDS})
            mlflow.log_metrics({"cv_mae": record["cv_mae"], "cv_rmse": record["cv_rmse"], "cv_mae_std": record["cv_mae_std"]})
    except Exception as exc:
        logger.warning("Remote MLflow logging failed for trial %s; local CSV is safe: %s", trial_no, exc)

    logger.info("Finished trial %s/%s: mean CV MAE %.3f", trial_no, len(candidates), record["cv_mae"])

trial_table = pd.DataFrame(trial_records).sort_values(["cv_mae", "cv_rmse"]).reset_index(drop=True)
trial_table.to_csv(RUN_DIR / "cv_trials_sorted.csv", index=False)
display(trial_table.round(3))

## 6. Evaluate the best CV configuration on our **original held-out validation engines**

This is the first time these engines are used for the tuned model. The earlier notebook reported the following **rounded** validation results on the same split:

- Previous combined approach: **MAE 32.918, RMSE 46.383**
- Four individual models: **MAE 32.941, RMSE 46.903**

We use these as comparison references, not as training targets. Do **not** use NASA test results to choose the winner.

In [ ]:
# Choose hyperparameters using cross-validation scores ONLY.
best_trial = trial_table.iloc[0].to_dict()
best_params = {key: best_trial[key] for key in search_space}
best_params["n_estimators"] = int(best_params["n_estimators"])
best_params["max_depth"] = int(best_params["max_depth"])
best_params["min_child_weight"] = int(best_params["min_child_weight"])

# Train on the original 80%-engine training split; score on the held-out 20%.
tuned_validation_model = build_xgb(best_params)
tuned_validation_model.fit(X_train, y_train)
val_predictions = tuned_validation_model.predict(X_val)
tuned_val = score_predictions(y_val, val_predictions)

comparison = pd.DataFrame([
    {"approach": "Previous combined model", "val_mae": 32.918, "val_rmse": 46.383},
    {"approach": "Four individual models", "val_mae": 32.941, "val_rmse": 46.903},
    {"approach": "Tuned combined XGBoost", "val_mae": tuned_val["mae"], "val_rmse": tuned_val["rmse"]},
]).sort_values(["val_mae", "val_rmse"]).reset_index(drop=True)

comparison.to_csv(RUN_DIR / "validation_comparison.csv", index=False)
joblib.dump(tuned_validation_model, RUN_DIR / "tuned_validation_model.joblib")
logger.info("Tuned validation MAE: %.3f | RMSE: %.3f", tuned_val["mae"], tuned_val["rmse"])
display(comparison.round(3))

## 7. Select the deployment approach and save a deployable model

**Selection rule:** Lowest pooled validation MAE. If tuning does not beat the previous combined model, keep the previous model; do not call the new model the winner.

This cell saves a **new combined XGBoost candidate** refitted on all four original training datasets, plus metadata for the CSV-upload API. It does **not** overwrite your previous `models/champion/` files or register a model.

If the tuned model is not the validation winner, the output says so clearly.

In [ ]:
# Determine whether tuning actually improved the previous combined baseline.
tuned_is_best = tuned_val["mae"] < min(32.918, 32.941)
selected_approach = "tuned_combined_xgboost" if tuned_is_best else "previous_combined_model"
logger.info("Selected using validation MAE: %s", selected_approach)

# Refit the tuned candidate on ALL original training engines for possible deployment.
# Do not use the NASA test dataset for this refit.
full_train = pd.concat([raw_train[fd].assign(
    **{f"dataset_{name}": int(fd == name) for name in DATASETS}
) for fd in DATASETS], ignore_index=True)
X_full = full_train[feature_columns]
y_full = full_train["RUL"]
deployment_candidate = build_xgb(best_params)
deployment_candidate.fit(X_full, y_full)

# Save a local model and exact input schema, without replacing prior champions.
candidate_path = RUN_DIR / "combined_xgboost_tuned_full_train.joblib"
joblib.dump(deployment_candidate, candidate_path)
metadata = {
    "candidate_type": "combined_xgboost_tuned",
    "selected_using_validation": selected_approach,
    "model_file": candidate_path.name,
    "dataset_ids": DATASETS,
    "input_features": base_features,
    "model_features": feature_columns,
    "indicator_features": indicator_features,
    "best_params": best_params,
    "validation_metrics": tuned_val,
    "cv_best_mae": float(best_trial["cv_mae"]),
    "training_rows": len(full_train),
    "test_set_used": False,
    "warning": "Joblib files must be loaded only from trusted sources.",
}
(RUN_DIR / "model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
logger.info("Saved candidate: %s", candidate_path)

## 8. Log final comparison and ONE model artifact to DagsHub MLflow

We upload only the **new tuned candidate**, not all 12 trial models. This avoids the long upload delays you experienced before.

MLflow may take a few minutes to upload this single artifact. The model and metrics are **already saved locally**, so a network failure will not erase them.

**No model registration yet.** We will register the actual winner after reviewing the comparison.

In [ ]:
# A separate summary run holds the comparison, exact parameters, and trained candidate.
try:
    with mlflow.start_run(run_name=f"{RUN_NAME}_final_comparison") as final_run:
        mlflow.set_tags({
            "stage": "validation_comparison",
            "approach": "combined",
            "selected_approach": selected_approach,
            "selection_metric": "validation_mae",
            "local_run_name": RUN_NAME,
        })
        mlflow.log_params({**best_params, "cv_folds": CV_FOLDS, "random_state": SEED})
        mlflow.log_metrics({
            "best_cv_mae": float(best_trial["cv_mae"]),
            "tuned_val_mae": tuned_val["mae"],
            "tuned_val_rmse": tuned_val["rmse"],
            "previous_combined_val_mae": 32.918,
            "previous_individual_val_mae": 32.941,
        })
        mlflow.log_artifact(str(TRIALS_CSV), artifact_path="reports")
        mlflow.log_artifact(str(RUN_DIR / "validation_comparison.csv"), artifact_path="reports")
        mlflow.log_artifact(str(RUN_DIR / "model_metadata.json"), artifact_path="model")
        mlflow.log_artifact(str(candidate_path), artifact_path="model")
        logger.info("DagsHub MLflow summary run: %s", final_run.info.run_id)
except Exception as exc:
    logger.warning("Remote summary upload failed; local model and reports remain saved: %s", exc)

print("Local results:", RUN_DIR)
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Validation winner:", comparison.iloc[0]["approach"])
print("Tuned combined MAE:", round(tuned_val["mae"], 3))
print("Tuned combined RMSE:", round(tuned_val["rmse"], 3))

## Next: final test and deployment

**Stop here and review the validation comparison.** If the tuned combined model wins, we can use the saved `combined_xgboost_tuned_full_train.joblib` and `model_metadata.json` for a FastAPI CSV-upload website. If it doesn't win, we should not automatically deploy it.

We have **not** used the NASA test sets in this tuning notebook. After choosing the final deployment approach, evaluate the locked model on the final test set and report that result without using it to adjust hyperparameters.

**Dependencies:** `pandas`, `pyarrow`, `numpy`, `scikit-learn`, `xgboost`, `joblib`, `mlflow`, `dagshub` (add any missing packages to your project's `requirements.txt`).